# Import

In [7]:
import pandas as pd
import plotly.express as px

In [8]:
# load data_all
data_all = pd.read_csv("../resources/final/data_all.csv")

# add gnd info to authors via gndIdentifier
gnd_all = pd.read_csv("../resources/gnd_all_mit_namensvarianten.csv", sep=";")
gnd_add = pd.read_csv("../resources/gnd_add.csv")
gnd_add['gndIdentifier'] = gnd_add['GND']
gnd = pd.concat([gnd_all, gnd_add])
data_all = data_all.merge(gnd[['gndIdentifier', 'GND_gender', 'GND_country', 'GND_birth', 'GND_death']], on = 'gndIdentifier', how = 'left')
data_all.query("GND_gender.isna() and gndIdentifier!='NoID'")

,corpus,author_first,author_last,filename,title,year,gndIdentifier,GND_name,gnd_match_status,canonicity_score,canonicity_rank,canonicity_orig,gndIdentifier_orig,correction,GND_gender,GND_country,GND_birth,GND_death


In [9]:
data_all.query("corpus=='12_diskoclassics'").sample(n=10)

,corpus,author_first,author_last,filename,title,year,gndIdentifier,GND_name,gnd_match_status,canonicity_score,canonicity_rank,canonicity_orig,gndIdentifier_orig,correction,GND_gender,GND_country,GND_birth,GND_death
19061,12_diskoclassics,Herman,Melville,NaN,Taipi,1846.0,118580604,"Melville, Herman",lobid_match,NaN,NaN,NaN,118580604,NaN,Männlich,USA,1819-08-01,1891-09-28
19043,12_diskoclassics,NaN,Anonym,NaN,Der Liebe Lust und Leid der Frau zur Frau,1895.0,NoID,NaN,anonymous,NaN,NaN,NaN,NoID,NaN,NaN,NaN,NaN,NaN
19079,12_diskoclassics,NaN,Voltaire,NaN,Candide,1782.0,118627813,Voltaire,exact,NaN,NaN,NaN,118627813,NaN,Männlich,"Großbritannien, Deutschland, Frankreich",1694.0,1778.0
19078,12_diskoclassics,Jean-Baptiste,Louvet de Couvray,NaN,Leben und Abenteuer des Chevalier Faublas – Dr...,1789.0,118881485,"Louvet de Couvray, Jean-Baptiste",lobid_match,NaN,NaN,NaN,118881485,NaN,Männlich,Frankreich,1760-06-12,1797-08-25
19006,12_diskoclassics,Hans,Siemens,NaN,Gymnasiast,1923.0,1035155125,"Siemens, Hans Peter",lobid_match,NaN,NaN,NaN,1035155125,NaN,Männlich,Deutschland,1879-09-05,1961-10-26
19052,12_diskoclassics,Wilkie,Collins,NaN,Frau und Gesetz,1875.0,118638432,"Collins, Wilkie",lobid_match,NaN,NaN,NaN,118638432,NaN,Männlich,Großbritannien,1824-01-08,1889-09-23
19024,12_diskoclassics,Jules,Barbey d'Aurevilly,NaN,Vom Dandytum und von G. Brummell,1909.0,118506498,"Barbey d'Aurevilly, Jules Amédée",lobid_match,NaN,NaN,NaN,118506498,NaN,Männlich,Frankreich,1808-11-02,1889-04-23
19062,12_diskoclassics,George,Sand,NaN,Gabriel,1839.0,118605348,"Sand, George",lobid_match,NaN,NaN,NaN,118605348,NaN,Weiblich,Frankreich,1804-07-05,1876-06-08
19075,12_diskoclassics,Friedrich,Schlegel,NaN,Lucinde,1799.0,118607987,"Schlegel, Friedrich von",exact,96.721325,42.0,NaN,118607987,NaN,Männlich,"Deutschland, Frankreich, Österreich",1772.0,1829.0
19008,12_diskoclassics,Frank,Heller,NaN,Des Kaisers alte Kleider,1920.0,118844865,"Heller, Frank",lobid_match,NaN,NaN,NaN,118844865,NaN,Männlich,Schweden,1886-07-20,1947-10-14


# Analysis

In [10]:
overview = []
for corpus in data_all['corpus'].unique():
    corpus_df = data_all.query("corpus==@corpus").copy()
    if corpus == '10_19C-prose':
        corpus_df = corpus_df.drop_duplicates(subset=['author_first', 'author_last', 'title'])
    corpus_df['canonicity_score'] = corpus_df['canonicity_score'].fillna(0)

    corpus_len = len(corpus_df)

    overview.append({
        'corpus' : corpus,
        'n_work' : corpus_len,

        'mean_canonicity_score' : corpus_df['canonicity_score'].mean(),
        '1–49' : len(corpus_df.query("1 <= canonicity_rank <= 49"))/corpus_len,
        '50–199' : len(corpus_df.query("50 <= canonicity_rank <= 199"))/corpus_len,
        '200–499' : len(corpus_df.query("200 <= canonicity_rank <= 499"))/corpus_len,
        '500–999' : len(corpus_df.query("500 <= canonicity_rank <= 999"))/corpus_len,
        '1000 and lower' : len(corpus_df.query("1000 <= canonicity_rank <= 99999"))/corpus_len,
        'no rank' : len(corpus_df.query("canonicity_rank.isna()"))/corpus_len,

        'nonmale_share' : len(corpus_df.query("GND_gender!='Männlich'"))/corpus_len,
    })

overview_df = pd.DataFrame(overview)
overview_df

,corpus,n_work,mean_canonicity_score,1–49,50–199,200–499,500–999,1000 and lower,no rank,nonmale_share
0,01_german-fiction,2735,37.439981,0.171846,0.119196,0.153931,0.158684,0.281536,0.114808,0.118099
1,01_translated-fiction,484,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
2,03_de-corp,4215,25.008974,0.104152,0.095848,0.126216,0.128826,0.339027,0.205931,0.171293
3,03_de-corp-non-fiction,765,20.878934,0.074510,0.018301,0.062745,0.109804,0.214379,0.520261,0.117647
4,04_d-prose,2511,18.169506,0.077260,0.158104,0.187575,0.174034,0.287535,0.115492,0.137794
5,05_ELTeC-deu,100,48.360014,0.250000,0.190000,0.240000,0.150000,0.170000,0.000000,0.330000
6,06_romane-der-romantik,115,92.326355,0.339130,0.121739,0.200000,0.173913,0.147826,0.017391,0.330435
7,07_bildungsromane,220,74.653681,0.295455,0.118182,0.159091,0.113636,0.186364,0.127273,0.313636
8,08_roko,1834,16.458475,0.070883,0.086696,0.149400,0.141767,0.368048,0.183206,0.242094
9,09_ecocor,190,59.646779,0.221053,0.094737,0.226316,0.121053,0.257895,0.078947,0.431579


In [11]:
rank_cols = [
    '1–49', '50–199', '200–499',
    '500–999', '1000 and lower', 'no rank'
]

df_plot = (
    overview_df
    .query("corpus!='01_translated-fiction' and corpus!='03_de-corp-non-fiction'")
    .sort_values(by='mean_canonicity_score')
)

df_plot['corpus'] = df_plot['corpus'].replace({
    '01_german-fiction' : 'Corpus of German Language Fiction',
    '03_de-corp' : 'de-Corp',
    '04_d-prose' : 'd-prose',
    '05_ELTeC-deu' : 'ELTeC-deu',
    '06_romane-der-romantik' : 'd-RoRo',
    '07_bildungsromane' : 'Bildungsromankorpus',
    '08_roko' : 'Romankorpus',
    '09_ecocor' : 'German EcoCor',
    '10_19C-prose' : '19C prose',
    '11_dta-belletristik' : 'DTA Kernkorpus Belletristik',
    '12_diskoclassics' : 'Disko Classics'
})

# (a) red -> green, one color per segment
colors = px.colors.sample_colorscale('RdYlGn', [i / (len(rank_cols) - 1) for i in range(len(rank_cols))])

fig = px.bar(
    df_plot,
    x='corpus',
    y=rank_cols,
    color_discrete_sequence=colors,
)

# (b) n_work above each stacked bar
totals = df_plot[rank_cols].sum(axis=1)
for corpus, total, n in zip(df_plot['corpus'], totals, df_plot['n_work']):
    fig.add_annotation(
        x=corpus, y=total,
        text=f"{n:,}",
        showarrow=False,
        yshift=10,
    )

# (c) mean canonicity score under each bar, as a second line of the tick label
fig.update_xaxes(
    tickmode='array',
    tickvals=df_plot['corpus'],
    ticktext=[
        f"{c}<br><span style='color:grey'>Ø {s:.2f}</span>"
        for c, s in zip(df_plot['corpus'], df_plot['mean_canonicity_score'])
    ],
)

fig.update_layout(
    margin=dict(t=60),
    yaxis_title='share of works',
    xaxis_title=None,
    legend_title='canonicity rank'
)
fig.show()

In [12]:
px.scatter(
    df_plot,
    x = 'mean_canonicity_score',
    y = 'nonmale_share',
    hover_name='corpus',
)